<a href="https://colab.research.google.com/github/lzx1990/lzx1990/blob/main/ee-python/large_gridded_exports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Large Gridded Exports from GEE

This notebook shows how to export a country-scale raster from Earth Engine as separate tiles using a grid. The key is to ensure use of `crs` and `crsTransform` to ensure all the tiles are in the same pixel grid and align correctly with the target projection.

[Read the full post](https://spatialthoughts.com/2024/10/23/large-image-exports-gee/)

In [1]:
import geemap
import ee

#### Initialization

First of all, you need to run the following cells to initialize the API and authorize your account. You must have a Google Cloud Project associated with your GEE account. Replace the `cloud_project` with your own project from [Google Cloud Console](https://console.cloud.google.com/).

In [2]:
# Replace the cloud_project with your own project
cloud_project = 'ee-longzexu1990'
try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

In [13]:
import ee

# 定义矩形的经纬度范围
min_lon = 95;
min_lat = 21;
max_lon = 117;
max_lat = 37;

# 创建一个矩形几何对象
geometry = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat]);

# 打印几何对象的信息（可选）
print('Rectangular Geometry:', geometry.getInfo());

m = geemap.Map()
m.addLayer(geometry, {'color': 'FF0000'}, 'Geometry Rectangle')
m.centerObject(geometry, 10)
m

Rectangular Geometry: {'type': 'Polygon', 'coordinates': [[[95, 21], [117, 21], [117, 37], [95, 37], [95, 21]]]}


Map(center=[29.2280216681916, 106.00000000000001], controls=(WidgetControl(options=['position', 'transparent_b…

#### Data Prep

We select a country and create a clipped ESA WorldCover 2020 classification for the region.

In [14]:
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate('2020-01-01', '2021-01-01').filterBounds(geometry)

# Extract the projection of the first band of the first image
embeddingsProjection = ee.Image(embeddings.first()).select(0).projection();
print(embeddingsProjection)
# mosaic different embeddings
image = embeddings.mosaic().setDefaultProjection(embeddingsProjection)


ee.Projection({
  "functionInvocationValue": {
    "functionName": "Image.projection",
    "arguments": {
      "image": {
        "functionInvocationValue": {
          "functionName": "Image.select",
          "arguments": {
            "bandSelectors": {
              "constantValue": [
                0
              ]
            },
            "input": {
              "functionInvocationValue": {
                "functionName": "Collection.first",
                "arguments": {
                  "collection": {
                    "functionInvocationValue": {
                      "functionName": "Collection.filter",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.filter",
                            "arguments": {
                              "collection": {
                                "functionInvocationValue": {
                      

#### Create a Grid

We create a grid and calculate the parameters for the CRS Transform. Each tile of the grid will be exported as a separate image on the chosen pixel grid.

In [15]:
crs = image.select(0).projection().crs()

# Choose the pixel size for export (meters)
pixelSize = 1000

# Choose the export tile size (pixels)
tileSize = 100

# Calculate the grid size (meters)
gridSize = tileSize * pixelSize

# Create the grid covering the geometry bounds
bounds = geometry.bounds(**{
  'proj': crs, 'maxError': 1
})

grid = bounds.coveringGrid(**{
  'proj':crs, 'scale': gridSize
})


#### Calculate the CRS Transform

In [16]:
# Calculate the coordinates of the top-left corner of the grid
bounds = grid.geometry().bounds(**{
  'proj': crs, 'maxError': 1
});

# Extract the coordinates of the grid
coordList = ee.Array.cat(bounds.coordinates(), 1)

xCoords = coordList.slice(1, 0, 1)
yCoords = coordList.slice(1, 1, 2)

# We need the coordinates of the top-left pixel
xMin = xCoords.reduce('min', [0]).get([0,0])
yMax = yCoords.reduce('max', [0]).get([0,0])

# Create the CRS Transform

# The transform consists of 6 parameters:
# [xScale, xShearing, xTranslation,
#  yShearing, yScale, yTranslation]
transform = ee.List([
    pixelSize, 0, xMin, 0, -pixelSize, yMax]).getInfo()
print(transform)

[1000, 0, -600000, 0, -1000, 4200000]


#### Resample or Aggregate Pixels

By default, the images are resampled to the target pixel grid using the Nearest Neighbor method. This is fine for most types of images, but you may want to change this behavior for certain types of operations. For discrete rasters, such as landcover classification, nearest neighbor is appropriate. For climate or elevation rasters, you may want to enable `bilinear` or `bicubic` interpolation.



In [17]:



## Aggregate pixels with 'mean' statistics
imageResampled = image.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=20000
      ).reproject(
    crs=crs,
    scale=1000);


#### Set a NoData Value

This is an important step. If you have masked pixels in your image, the output tiles will not be of equal size. To ensure each tile has the same dimensions and there are no gaps or overlapping pixels, `unmask()` all masked pixels and set a nodata value.

In [18]:
# Assign a no-data value
noDataValue = 0
exportImage = imageResampled.unmask(**{
    'value':noDataValue,
    'sameFootprint': False
})

#### Export Tiles

We created the tiling grid using the bounding box of the region geometry. This may result in certain grids that have no overlap with the region and thus will be empty. We can filter out those empty grids before exporting.

In [19]:
filtered_grid = grid.filter(ee.Filter.intersects('.geo', geometry))
m.addLayer(filtered_grid, {'color': 'red'}, 'Filtered Grid')
m

Map(bottom=2127.0, center=[19.07509724212452, 135.61945946758422], controls=(WidgetControl(options=['position'…

In [20]:
tile_ids = filtered_grid.aggregate_array('system:index').getInfo();
print('Total tiles', len(tile_ids))

Total tiles 429


In [21]:
# Export each tile
# Warning: This will result in 14 large GeoTIFFs tiles in your Google Drive
for i, tile_id in enumerate(tile_ids):
    feature = ee.Feature(filtered_grid.toList(1, i).get(0))
    geometry = feature.geometry()
    task_name = 'tile_' + tile_id.replace(',', '_')
    task = ee.batch.Export.image.toDrive(**{
        'image': exportImage.select([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]),
        'description': f'Image_Export_{task_name}',
        'fileNamePrefix': task_name,
        'folder':'earthengine0507',
        'crs': crs,
        'crsTransform': transform,
        'region': geometry,
        'maxPixels': 1e13
    })
    task.start()
    print('Started Task: ', i+1)

Started Task:  1
Started Task:  2
Started Task:  3
Started Task:  4
Started Task:  5
Started Task:  6
Started Task:  7
Started Task:  8
Started Task:  9
Started Task:  10
Started Task:  11
Started Task:  12
Started Task:  13
Started Task:  14
Started Task:  15
Started Task:  16
Started Task:  17
Started Task:  18
Started Task:  19
Started Task:  20
Started Task:  21
Started Task:  22
Started Task:  23
Started Task:  24
Started Task:  25
Started Task:  26
Started Task:  27
Started Task:  28
Started Task:  29
Started Task:  30
Started Task:  31
Started Task:  32
Started Task:  33
Started Task:  34
Started Task:  35
Started Task:  36
Started Task:  37
Started Task:  38
Started Task:  39
Started Task:  40
Started Task:  41
Started Task:  42
Started Task:  43
Started Task:  44
Started Task:  45
Started Task:  46
Started Task:  47
Started Task:  48
Started Task:  49
Started Task:  50
Started Task:  51
Started Task:  52
Started Task:  53
Started Task:  54
Started Task:  55
Started Task:  56
S